# 3차 개인 기준선·패턴 검증
Purpose: compare G0, selected personal baselines, P2+CL, and P2+CL-B sequentially on validation people.

기존 검증 저장 순서는 01 → 03 → 02 → 04입니다. 05가 COMPLETE 및 Save Version 된 뒤 이 06을 단독 실행합니다.

> Warning: this is an oracle/sanity-only synthetic-data benchmark. Behavior labels remain auxiliary and the locked test is not opened.

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import pandas as pd

SERIES_ID = "mvp3-oracle-v1"
EXPECTED_SPLIT_COUNTS = {"train": 24, "validation": 6, "locked_test": 6}
DATA_STATUS = "oracle/sanity"
REAL_ACCURACY_STATUS = "NOT VERIFIED"
DEVICE_SYNCHRONIZATION_STATUS = "NOT_AVAILABLE_TRUTH_ONLY"
RUN_TRAINING = False
RUN_LOCKED_TEST = False
RUN_PHASE3_VALIDATION = True
CANDIDATE_WITH_LIMITED_LOAD_BASELINE = "P2+CL-B"
FEATURE_WINDOWS_SECONDS = (30,)
FEATURE_LAGS_SECONDS = (1,)
RESOURCE_PROFILE = "reduced-memory-phase3-v1"
OUTPUT_ROOT = Path("/kaggle/working/phase3_result")


def discover_source() -> tuple[Path, Path, Path]:
    input_root = Path("/kaggle/input")
    source = next(input_root.rglob("phase3_source.parquet"))
    splits = next(input_root.rglob("splits.parquet"))
    wheel = next(
        path
        for path in input_root.rglob("multisensor_ml-*.whl")
        if "multisensor-goal1-5-phase3-raw-source" in str(path)
    )
    return source, splits, wheel


if RUN_PHASE3_VALIDATION:
    source_path, splits_path, wheel_path = discover_source()
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--no-deps", str(wheel_path)],
        check=True,
    )
    from multisensor_ml.phase3_pipeline import (
        build_locked_identity_stubs,
        run_phase3_experiment,
    )

    source = pd.read_parquet(source_path)
    locked = pd.read_parquet(splits_path).loc[lambda value: value["split_role"].eq("locked_test")]
    locked_stub = build_locked_identity_stubs(source, locked)
    combined = pd.concat([source, locked_stub], ignore_index=True)
    outputs = run_phase3_experiment(
        combined,
        OUTPUT_ROOT,
        windows=FEATURE_WINDOWS_SECONDS,
        lags=FEATURE_LAGS_SECONDS,
    )
    manifest = json.loads(outputs.manifest_json.read_text())
    manifest["resource_profile"] = RESOURCE_PROFILE
    manifest["feature_windows_sec"] = list(FEATURE_WINDOWS_SECONDS)
    manifest["feature_lags_sec"] = list(FEATURE_LAGS_SECONDS)
    outputs.manifest_json.write_text(json.dumps(manifest, indent=2, sort_keys=True))
    assert manifest["locked_test_read"] is False
    assert manifest["candidate_ids"][-1] == CANDIDATE_WITH_LIMITED_LOAD_BASELINE
    assert outputs.candidate_metrics_parquet.name == "phase3_candidate_metrics.parquet"
    print(json.dumps(manifest, indent=2, ensure_ascii=False))